In [90]:
import pandas as pd

df = pd.read_csv("../../data/processed/red_wine_cleaned.csv")

In [91]:
display(df.head())
display(df.shape)
df.info()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.66,0.00,1.8,0.075,13.0,40.0,0.9978,3.51,0.56,9.4,5


(1359, 12)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1359 entries, 0 to 1358
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1359 non-null   float64
 1   volatile acidity      1359 non-null   float64
 2   citric acid           1359 non-null   float64
 3   residual sugar        1359 non-null   float64
 4   chlorides             1359 non-null   float64
 5   free sulfur dioxide   1359 non-null   float64
 6   total sulfur dioxide  1359 non-null   float64
 7   density               1359 non-null   float64
 8   pH                    1359 non-null   float64
 9   sulphates             1359 non-null   float64
 10  alcohol               1359 non-null   float64
 11  quality               1359 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 127.5 KB


In [92]:
X = df.drop("quality", axis=1)
y = df["quality"]

In [93]:
# Train/Test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# Linear Regression

In [94]:
# Import module
from sklearn.linear_model import LinearRegression

# Create the model
model = LinearRegression()

# Train the model 
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

In [95]:
# Evaluate
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)

print(f"R²: {r2_score(y_test, y_pred):.3f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.3f}")
print(f"RMSE: {root_mean_squared_error(y_test, y_pred):.3f}")

R²: 0.392
MAE: 0.504
RMSE: 0.657


### Baseline Linear Regression Model

A baseline **Linear Regression** model was trained using the physicochemical properties as predictor variables and the wine quality score as the target variable. The dataset was divided into training and testing sets using an 80/20 split.

The model achieved an **R² score of 0.392** on the test set.

This indicates that the model explains approximately **39% of the variation** in wine quality. While the model captures some relationship between the chemical properties and wine quality, a substantial amount of the variability remains unexplained.

This baseline provides a useful point of comparison for more advanced machine learning models. Because wine quality is likely influenced by complex, nonlinear relationships among the features, tree-based ensemble methods such as Random Forests or Gradient Boosting may achieve better predictive performance.


# Scaling

In [96]:
from sklearn.preprocessing import StandardScaler

# Create the scaler
scaler = StandardScaler()

# Learn the mean and standard deviation from the training data
X_train_scaled = scaler.fit_transform(X_train)

# Apply the same transformation to the test data
X_test_scaled = scaler.transform(X_test)

In [97]:
# Train the model using the scaled data
model.fit(X_train_scaled, y_train)

# Predict
y_pred = model.predict(X_test_scaled)

# Evaluate
print(f"R²: {r2_score(y_test, y_pred):.3f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.3f}")
print(f"RMSE: {root_mean_squared_error(y_test, y_pred):.3f}")

R²: 0.392
MAE: 0.504
RMSE: 0.657


#### Feature Scaling Evaluation

The predictor variables were standardized using `StandardScaler` and a Linear Regression model was retrained using the scaled features.

The scaled and unscaled models produced virtually identical performance metrics indicating that feature scaling did not improve the performance of the Linear Regression model. This outcome is expected because ordinary least squares Linear Regression is generally insensitive to the scale of the predictor variables.

Therefore, the original feature set was retained for the baseline model.

# Decision Tree

In [98]:
from sklearn.tree import DecisionTreeRegressor

model = DecisionTreeRegressor(random_state=42)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(f"R²: {r2_score(y_test, y_pred):.3f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.3f}")
print(f"RMSE: {root_mean_squared_error(y_test, y_pred):.3f}")

R²: -0.261
MAE: 0.651
RMSE: 0.945


#### Decision Tree Regression Discussion

The Decision Tree Regressor performed significantly worse than the baseline Linear Regression model. A negative R² score indicates that the model performed worse than predicting the mean wine quality for all observations.

This result suggests that the default decision tree overfit the training data and did not generalize well to the unseen test data.


# Random Forest Regressor

In [99]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(random_state=42)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(f"R²: {r2_score(y_test, y_pred):.3f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.3f}")
print(f"RMSE: {root_mean_squared_error(y_test, y_pred):.3f}")

R²: 0.458
MAE: 0.468
RMSE: 0.619


#### Random Forest Regressor Model Discussion

The Random Forest Regressor achieved a better performance than the base model. It produced a higher R² score and lower MAE and RMSE values, indicating that it explained more of the variability in wine quality while making more accurate predictions than the other models.

These results suggest that the relationship between the physicochemical properties and wine quality is not purely linear. The Random Forest model is able to capture more complex, nonlinear relationships than the Linear Regression model.


# Gradient Boosting Regressor

In [100]:
from sklearn.ensemble import GradientBoostingRegressor

model = GradientBoostingRegressor(random_state=42)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(f"R²: {r2_score(y_test, y_pred):.3f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.3f}")
print(f"RMSE: {root_mean_squared_error(y_test, y_pred):.3f}")

R²: 0.459
MAE: 0.478
RMSE: 0.619


#### Gradient Boosting Regressor Discussion

A Gradient Boosting Regressor was trained using the original physicochemical features from the cleaned red wine dataset. Gradient Boosting builds an ensemble of decision trees sequentially, where each new tree attempts to correct the errors made by the previous trees.

The model was evaluated using R², Mean Absolute Error (MAE), and Root Mean Squared Error (RMSE).

The Gradient Boosting Regressor achieved an R² score of 0.459, indicating that the model explains approximately 46% of the variation in wine quality. The model produced an average prediction error of approximately 0.48 quality points based on the MAE metric.

Compared with the baseline Linear Regression model, Gradient Boosting produced improved performance, with a higher R² score and lower MAE and RMSE values. This suggests that the model was better able to capture nonlinear relationships between the physicochemical properties and wine quality.


# Final Result
| Model                       | Preprocessing   |        R² |       MAE |      RMSE |
| --------------------------- | --------------- | --------: | --------: | --------: |
| Linear Regression           | None (Baseline) |     0.392 |     0.504 |     0.657 |
| Linear Regression           | StandardScaler  |     0.392 |     0.504 |     0.657 |
| Decision Tree Regressor     | None            |    -0.261 |     0.651 |     0.945 |
| Random Forest Regressor     | None            |     0.458 | **0.468** |     0.619 |
| Gradient Boosting Regressor | None            | **0.459** |     0.478 |     0.619 |

#### Conclusion
Feature scaling did not change the performance of the Linear Regression model. The R², MAE, and RMSE values remained unchanged, indicating that scaling the predictor variables had no measurable effect on this baseline model.  
The Decision Tree Regressor performed substantially worse than the baseline, suggesting it overfit the training data.  
The Random Forest Regressor and Gradient Boosting Regressor both outperformed the Linear Regression model. Their performance was very similar, with Gradient Boosting achieving the highest R² and Random Forest producing the lowest MAE.  
The results suggest that ensemble tree-based methods are better able to capture the nonlinear relationships between the wine's physicochemical properties and its quality than a simple linear model.